# 06 · Linear GAM: results and comparison

This notebook fits nothing and scores nothing. It reads what the scripts wrote and plots it:

1. `python scripts/evaluate_linear_gam.py` validates `linear_gam` and `lr_mos` (the LR on the same inputs), saves the final models and writes the frozen Sep 17–30 forecasts.
2. `python scripts/score_submission.py reports/predictions/linear_gam.csv reports/predictions/lr_mos.csv` scores them on the test period, once (needed only for Section 5).

The model is described in `docs/linear_gam.md`: same approach as the LR (anomaly target, one model per lead day, recency weights, the same 12 inputs), with a smooth curve per input instead of a straight line. Section 4 needs `pygam` installed to load the saved GAMs.

In [ ]:
import hashlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from weather_modeling import config
from weather_modeling.dataset import TARGET, load_forecast_features
from weather_modeling.evaluation import plot_by_lead_day
from weather_modeling.metrics import bootstrap_mae_difference

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 30)
REPORTS = config.PROJECT_ROOT / "reports"
MODELS = ["lr_mos", "linear_gam"]
COMPARE = ["persistence", "climatology", "raw_ecmwf", *MODELS]
COLORS = {"climatology": "#2a78d6", "persistence": "#eb6834", "raw_ecmwf": "#1baf7a", "lr_mos": "#4a3aa7", "linear_gam": "#008300"}


def report(name):
    path = REPORTS / name
    if not path.exists():
        raise FileNotFoundError(f"{path.relative_to(config.PROJECT_ROOT)} is missing; run the script that writes it (see above)")
    return path

## 1. Validation folds

MAE by lead day on the two shared folds; `all` weights the 14 lead days equally. Each model was trained only on targets observed before the fold's first target.

In [ ]:
validation = pd.read_csv(report("linear_gam_validation.csv"), index_col=["metric", "fold", "lead_day"])
mae = validation.loc["mae"]
folds = [fold for fold in mae.index.unique("fold") if fold in ("autumn_2025", "late_summer_2026")]
display(mae.loc[folds].xs("all", level="lead_day")[COMPARE].round(2))

overall = pd.read_csv(report("linear_gam_overall.csv"), index_col=["fold", "weighting"])
display(overall[COMPARE].round(2))

equal = mae.loc[folds].xs("all", level="lead_day")
display(pd.DataFrame({b: 100 * (1 - equal["linear_gam"] / equal[b]) for b in ["persistence", "climatology", "raw_ecmwf", "lr_mos"]})
        .round(0).add_prefix("% lower MAE than "))

In [ ]:
for fold in folds:
    plot_by_lead_day(mae.loc[fold][["climatology", "raw_ecmwf", "persistence", *MODELS, "n"]], f"MAE by lead day, {fold}",
                     path=config.FIGURES_DIR / f"linear_gam_mae_{fold}.png")
    plt.show()

## 2. The six 2026 windows

Single forecasts set up like the real task, each from models retrained on data before the window. MAE with hours pooled.

In [ ]:
windows = pd.read_csv(report("linear_gam_windows.csv"), index_col="window")
display(windows[COMPARE].round(2))

shown = ["raw_ecmwf", *MODELS]
fig, ax = plt.subplots(figsize=(9, 3.4))
x = np.arange(len(windows))
for k, name in enumerate(shown):
    ax.bar(x + (k - 1) * 0.27, windows[name], width=0.27, color=COLORS[name], label=name)
ax.set_xticks(x, windows.index)
ax.set(ylabel="MAE (°C)", title="MAE of each 2026 window")
ax.legend(fontsize=8, frameon=False)
fig.tight_layout()
plt.show()

## 3. GAM vs LR, hour by hour

From the out-of-fold predictions: where in the day the two models differ, and whether the gap on each fold is larger than noise. The interval resamples 14-day blocks of runs, so with four or five blocks per fold it is a rough guide only.

In [ ]:
predictions = pd.read_csv(report("linear_gam_validation_predictions.csv.gz"), parse_dates=["run_time_utc", "valid_time_utc", "target_time_local"])
fold_rows = predictions[predictions["fold"].isin(folds)].dropna(subset=[TARGET, "raw_ecmwf", *MODELS])

fig, axes = plt.subplots(1, len(folds), figsize=(12, 3.4), sharey=True)
for ax, fold in zip(np.atleast_1d(axes), folds):
    part = fold_rows[fold_rows["fold"] == fold]
    by_hour = part[shown].sub(part[TARGET], axis=0).abs().groupby(part["hour_local"]).mean()
    for name in shown:
        ax.plot(by_hour.index, by_hour[name], marker="o", ms=3, color=COLORS[name], label=name)
    ax.set(title=fold, xlabel="Local hour", xticks=range(0, 24, 3))
np.atleast_1d(axes)[0].set(ylabel="MAE (°C)")
np.atleast_1d(axes)[0].legend(fontsize=8, frameon=False)
fig.tight_layout()
plt.show()

display(pd.DataFrame({fold: bootstrap_mae_difference(fold_rows[fold_rows["fold"] == fold], "linear_gam", "lr_mos")
                      for fold in folds}).T.round(3).rename(columns={"difference": "MAE(gam) - MAE(lr)"}))

## 4. What the final GAM learned

**Partial dependence**: each input's contribution to the predicted anomaly (°C), with a 95% band, at three lead days. A straight line means the LR's form was already right for that input; bends are what the GAM adds. Needs `pygam` to load the saved models.

In [ ]:
from weather_modeling.gam import load_models

saved = load_models()
gam_models, gam_features = saved["linear_gam"], saved["gam_features"]

SHOW_DAYS = (1, 7, 14)
terms = [i for i, term in enumerate(gam_models[1].terms) if not term.isintercept]
fig, axes = plt.subplots(len(SHOW_DAYS), len(terms), figsize=(2.3 * len(terms), 2.2 * len(SHOW_DAYS)), sharey="row")
for row, day in zip(axes, SHOW_DAYS):
    gam = gam_models[day]
    for ax, i in zip(row, terms):
        grid = gam.generate_X_grid(term=i)
        effect, band = gam.partial_dependence(term=i, X=grid, width=0.95)
        x = grid[:, gam.terms[i].feature]
        ax.plot(x, effect, color=COLORS["linear_gam"])
        ax.fill_between(x, band[:, 0], band[:, 1], color=COLORS["linear_gam"], alpha=0.15, lw=0)
        ax.axhline(0, color="grey", lw=0.5)
        if day == SHOW_DAYS[0]:
            ax.set_title(gam_features[gam.terms[i].feature], fontsize=8)
    row[0].set_ylabel(f"day {day}\n°C")
fig.tight_layout()
fig.savefig(config.FIGURES_DIR / "linear_gam_partial_dependence.png", dpi=150)
plt.show()

**How much each model follows ECMWF.** GAM: the average change of the prediction when both ECMWF anomaly inputs rise by 1 °C, over the last year of rows. LR: its two ECMWF coefficients added up. 1 = follow ECMWF, 0 = stay on climatology. Below: the smoothing GCV chose per lead day (higher effective dof = wigglier curves).

In [ ]:
weight = pd.read_csv(report("linear_gam_ecmwf_weight.csv"), index_col="lead_day")
fig, ax = plt.subplots(figsize=(9, 3.4))
for name in MODELS:
    ax.plot(weight.index, weight[name], marker="o", color=COLORS[name], label=name)
ax.axhline(0, color="grey", lw=0.6)
ax.set(title="Share of the ECMWF anomaly kept", xlabel="Lead day", ylabel="Weight", xticks=weight.index)
ax.legend(fontsize=8, frameon=False)
fig.tight_layout()
fig.savefig(config.FIGURES_DIR / "linear_gam_ecmwf_weight.png", dpi=150)
plt.show()

display(pd.read_csv(report("linear_gam_smoothing.csv"), index_col="lead_day").T.round(3))

## 5. The forecast for Sep 17–30

The frozen forecasts, checked against the fingerprints the script recorded, next to climatology and the raw ECMWF run they start from.

In [ ]:
frozen = pd.read_csv(config.PREDICTIONS_DIR / "frozen_forecasts.csv").set_index("name")["sha256"]
for name in MODELS:
    path = config.PREDICTIONS_DIR / f"{name}.csv"
    assert hashlib.sha256(path.read_bytes()).hexdigest() == frozen[name], f"{name}.csv changed since it was frozen"
forecast = load_forecast_features()[["target_time_local", "clim_temp_c", "ecmwf_t2m"]]
for name in MODELS:
    forecast[name] = pd.read_csv(config.PREDICTIONS_DIR / f"{name}.csv")["pred_temp_c"].to_numpy()

fig, ax = plt.subplots(figsize=(11, 3.6))
ax.plot(forecast["target_time_local"], forecast["clim_temp_c"], "--", lw=1, color=COLORS["climatology"], label="climatology")
ax.plot(forecast["target_time_local"], forecast["ecmwf_t2m"], lw=1, color=COLORS["raw_ecmwf"], label="raw ECMWF (Sep 16 12z)")
for name in MODELS:
    ax.plot(forecast["target_time_local"], forecast[name], lw=1.8 if name == "linear_gam" else 1.1, color=COLORS[name], label=name)
ax.set(title="Forecast for RDU, Sep 17-30, 2026 (local time)", ylabel="°C")
ax.legend(fontsize=8, frameon=False)
fig.tight_layout()
fig.savefig(config.FIGURES_DIR / "linear_gam_final_forecast.png", dpi=150)
plt.show()

## 6. Test period (after `score_submission.py`)

Read from `reports/final_scores.csv` and `reports/final_hourly.csv`. Results only: nothing here goes back into the models.

In [ ]:
scores = pd.read_csv(report("final_scores.csv"), index_col=["metric", "lead_day"])
hourly = pd.read_csv(report("final_hourly.csv"), parse_dates=["target_time_local"])
print(f"{hourly[TARGET].notna().sum()} of {len(hourly)} hours observed")
summary = pd.DataFrame({metric: scores.loc[(metric, "all"), COMPARE] for metric in ("mae", "rmse", "bias")})
pooled = hourly.dropna(subset=[TARGET, *COMPARE])
summary["mae, hours pooled"] = pooled[COMPARE].sub(pooled[TARGET], axis=0).abs().mean()
display(summary.round(2))

plot_by_lead_day(scores.loc["mae"][["climatology", "raw_ecmwf", "persistence", *MODELS]], "MAE by lead day, Sep 17-30, 2026",
                 path=config.FIGURES_DIR / "linear_gam_test_mae_by_lead_day.png")
plt.show()

fig, ax = plt.subplots(figsize=(11, 3.6))
ax.plot(hourly["target_time_local"], hourly[TARGET], color="black", lw=1.4, label="observed")
for name in ["raw_ecmwf", *MODELS]:
    ax.plot(hourly["target_time_local"], hourly[name], color=COLORS[name], lw=1.6 if name == "linear_gam" else 1, label=name)
ax.set(title="Sep 17-30, 2026: forecasts vs observed (local time)", ylabel="°C")
ax.legend(fontsize=8, frameon=False)
fig.tight_layout()
fig.savefig(config.FIGURES_DIR / "linear_gam_test_forecast_vs_observed.png", dpi=150)
plt.show()